# 2.nivelSegmento · 1. Creación de features (enfoque HÍBRIDO)

Extracción de features a **nivel de ventana (segmento)** en lugar de a nivel de sesión.
Cada fila del dataset resultante es una **ventana de habla del paciente**, con:

- **Features acústicas por ventana** (las mismas 45: F0, energía, 13 MFCC, espectrales,
  jitter/shimmer/HNR — media y std), que **varían ventana a ventana** → aquí está la
  ganancia de datos y de dinámica temporal.
- **Features de pausas/turnos de la sesión, difundidas** a todas las ventanas de esa sesión
  (mismo valor en todas): se conservan como **contexto**, porque las pausas son un concepto
  entre-segmentos que no existe dentro de una ventana. Así no se pierde el hallazgo de
  interpretabilidad más fuerte del trabajo (enlentecimiento psicomotor por pausas).

> **⚠️ Crítico para el modelado (no olvidar):** al haber varias ventanas por sesión, la
> validación cruzada DEBE particionar **por sesión** (`GroupKFold` / `StratifiedGroupKFold`
> con `groups=participant_id`). Si ventanas de una misma sesión caen en train y test a la vez,
> hay **fuga masiva** (pausas difundidas idénticas + acústica muy parecida) y las métricas
> saldrían infladas. La columna `participant_id` se conserva para esto.

Las funciones de extracción (`load_patient_segments`, `extract_spectral_features`,
`extract_praat_features`, `_functionals`) son **idénticas** a las del notebook de sesión
(`1.nivelSesion/2.CreacionFeatures_vf`), para que las features sean comparables.

## 0. Setup

In [1]:
import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

import librosa
import soundfile as sf
import parselmouth
from parselmouth.praat import call

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Ruta con df_sample.csv (salida del notebook 1.Carga_EDA_muestra) y destino de salida
path_output = r'C:/Users/tblxa/Desktop/Master/UNIR_IA_TFE/output'

# Speakers entrevista
name_paciente = 'Participant'
name_entrevistador = 'Ellie'

In [3]:
##################################################
#### Parámetros de extracción de features     ####
##################################################

SR          = 16_000            # Hz — estándar ASR/speech processing
N_MFCC      = 13                # coeficientes MFCC — estándar literatura (AVEC, ComParE)
FRAME_LEN   = int(0.025 * SR)  # 25 ms — ventana de análisis estándar ETSI
HOP_LEN     = int(0.010 * SR)  # 10 ms — paso entre frames
MIN_SEG_DUR = 0.3              # s — duración mínima de segmento (evita artefactos)
F0_FMIN     = 75.0             # Hz — mínimo F0 (voces masculinas graves)
F0_FMAX     = 400.0            # Hz — máximo F0 (voces femeninas agudas)
MIN_DUR_VQ  = 0.5              # s — mínimo para análisis de calidad de voz en Praat

##################################################
#### Parámetros de VENTANEO (nivel de segmento) ##
##################################################
WIN_DUR = 4.0    # s — duración de cada ventana
WIN_HOP = 4.0    # s — paso entre ventanas (= WIN_DUR → sin solape)
MIN_WIN = 2.0    # s — se conserva el resto final de un segmento si dura ≥ esto
VOICED_MIN = 0.05  # fracción sonora mínima para conservar una ventana (descarta silencio/ruido)

# Columnas de pausas/turnos (de df_sample) que se DIFUNDEN a todas las ventanas de la sesión.
# Son contexto de sesión: constantes dentro de cada sesión.
PAUSE_COLS = [
    'nTurns_pat', 'sumTurns_pat', 'avgTurns_pat', 'stdTurns_pat', 'ratioTurns_pat',
    'sumPause_inter_pat', 'avgPause_inter_pat', 'stdPause_inter_pat', 'ratioPause_inter_pat',
    'nPause_intra_pat', 'sumPause_intra_pat', 'avgPause_intra_pat', 'stdPause_intra_pat',
    'ratioPause_intra_pat',
]
META_COLS = ['participant_id', 'phq8_score', 'phq8_binary', 'gender', 'split']

In [4]:
df_sample = pd.read_csv(path_output + '/df_sample.csv')
print(f"Sesiones en df_sample: {len(df_sample)}")
df_sample.head(3)

Sesiones en df_sample: 186


,participant_id,phq8_binary,phq8_score,gender,split,path_audio,path_transcript,durAudio,durSession,nTurns_pat,...,ratioTurns_pat,sumPause_inter_pat,avgPause_inter_pat,stdPause_inter_pat,ratioPause_inter_pat,nPause_intra_pat,sumPause_intra_pat,avgPause_intra_pat,stdPause_intra_pat,ratioPause_intra_pat
0,300,0,2,1,test,D:/DAIZ-WOZ/base/300_P/300_AUDIO.wav,D:/DAIZ-WOZ/base/300_P/300_TRANSCRIPT.csv,648.5,584.68,58,...,0.309725,93.710,1.673393,1.786927,0.160276,29,25.330,0.873448,0.454503,0.043323
1,301,0,3,1,test,D:/DAIZ-WOZ/base/301_P/301_AUDIO.wav,D:/DAIZ-WOZ/base/301_P/301_TRANSCRIPT.csv,823.9,774.40,48,...,0.698438,37.070,0.823778,0.638970,0.047869,56,65.430,1.168393,0.526539,0.084491
2,302,0,4,1,dev,D:/DAIZ-WOZ/base/302_P/302_AUDIO.wav,D:/DAIZ-WOZ/base/302_P/302_TRANSCRIPT.csv,758.8,676.97,52,...,0.419564,97.925,1.958500,1.242940,0.144652,44,75.542,1.716864,0.914819,0.111588


## 1. Funciones de extracción (idénticas al nivel de sesión)
Reutilizadas verbatim del notebook de sesión para garantizar que las features son comparables.

In [5]:
## ─────────────────────────────────────────────────────────────────────────────
## 1. Funciones auxiliares
## ─────────────────────────────────────────────────────────────────────────────

# Media y desviación típica como únicos estadísticos funcionales.
# Justificación en la celda anterior.
_FUNCS = ['mean', 'std']


def _functionals(x: np.ndarray, prefix: str) -> dict:
    """
    Comprime un array 1D de valores frame-a-frame en media y desviación típica.

    Se usa std poblacional (ddof=0) para consistencia cuando el número de
    frames varía entre sesiones (numpy default).
    Retorna NaN si el array está vacío o es None.
    """
    if x is None or len(x) == 0:
        return {f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan}

    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std":  float(np.std(x)),
    }


def _empty_features() -> dict:
    """Retorna NaN para todas las features (sesiones con error)."""
    out = {}
    for prefix in ['f0', 'energy', 'centroid', 'bandwidth', 'rolloff', 'zcr',
                   'hnr', 'jitter', 'shimmer']:
        out[f"{prefix}_mean"] = np.nan
        out[f"{prefix}_std"]  = np.nan
    out['f0_voiced_frac'] = np.nan
    for i in range(1, N_MFCC + 1):
        out[f"mfcc{i:02d}_mean"] = np.nan
        out[f"mfcc{i:02d}_std"]  = np.nan
    return out


## ─────────────────────────────────────────────────────────────────────────────
## 2. Carga de segmentos del paciente
## ─────────────────────────────────────────────────────────────────────────────

def load_patient_segments(path_audio: str,
                           path_transcript: str,
                           speaker: str = 'Participant') -> list:
    """
    Carga el audio y extrae únicamente los segmentos de habla verbal del paciente.

    Decisión: aislar el habla del paciente via timestamps del transcript (ground
    truth anotado), en lugar de VAD automático. Esto evita contaminar las
    features con la voz de Ellie o los silencios interturno, que son
    acústicamente distintos a la habla del paciente y distorsionarían
    métricas prosódicas y de calidad vocal.

    Se excluyen marcadores no verbales (<laughter>, <cough>, <synch>, etc.)
    para no contaminar features acústicas con audio que no es habla.

    Se descartan segmentos < MIN_SEG_DUR (0.3 s) para evitar artefactos en
    el cálculo de MFCCs y estimación de F0.
    """
    df_t = pd.read_csv(path_transcript, sep='\t')
    df_t.columns = df_t.columns.str.strip()
    df_t = (df_t[df_t['speaker'] == speaker]
              .sort_values('start_time')
              .reset_index(drop=True))

    # Excluir marcadores no verbales: <laughter>, <cough>, <synch>, etc.
    df_t = df_t[~df_t['value'].str.strip().str.startswith('<', na=False)].reset_index(drop=True)

    y, _ = librosa.load(path_audio, sr=SR, mono=True)

    segments = []
    for _, row in df_t.iterrows():
        dur = row['stop_time'] - row['start_time']
        if dur < MIN_SEG_DUR:
            continue

        s = int(row['start_time'] * SR)
        e = int(row['stop_time']  * SR)
        seg = y[s:e]

        if len(seg) >= FRAME_LEN:
            segments.append(seg)

    return segments


## ─────────────────────────────────────────────────────────────────────────────
## 3. Features espectrales (librosa)
## ─────────────────────────────────────────────────────────────────────────────

def extract_spectral_features(segments: list) -> dict:
    """
    Features espectrales a nivel de frame sobre todos los segmentos del paciente.

    MFCCs (13 coeficientes):
        Capturan la forma del tracto vocal. Son la representación de referencia
        en reconocimiento de habla y detección de trastornos del habla.
        Los coeficientes bajos (1-4) son especialmente discriminativos para
        depresión (Williamson et al. 2016). Se usan 13 coeficientes (estándar
        MFCC basado en filterbank de 26 filtros Mel).

    Energía RMS (dB):
        Indicador de loudness. Los pacientes deprimidos muestran habla
        con menor energía y menor variabilidad energética.

    Centroide espectral:
        Centro de masa del espectro. Relacionado con el brillo tonal.
        Menor en voces más graves y monótonas (típico en depresión).

    Ancho de banda espectral:
        Dispersión del espectro en torno al centroide.

    Rolloff espectral (85%):
        Frecuencia por debajo de la cual se concentra el 85% de la energía.

    ZCR (Zero-Crossing Rate):
        Tasa de cruces por cero. Indicador de afonía y ronquera.

    Parámetros de análisis:
        - Ventana: 25 ms (estándar ETSI ES 201 108)
        - Hop: 10 ms (75% solapamiento)
        - FFT: misma longitud que ventana
    """
    all_mfcc     = []
    all_energy   = []
    all_centroid = []
    all_bw       = []
    all_rolloff  = []
    all_zcr      = []

    for seg in segments:
        if len(seg) < FRAME_LEN:
            continue

        all_mfcc.append(
            librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                  n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )

        rms = librosa.feature.rms(y=seg, frame_length=FRAME_LEN, hop_length=HOP_LEN)
        all_energy.append(20 * np.log10(rms + 1e-8))

        all_centroid.append(
            librosa.feature.spectral_centroid(y=seg, sr=SR,
                                               n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )
        all_bw.append(
            librosa.feature.spectral_bandwidth(y=seg, sr=SR,
                                                n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )
        all_rolloff.append(
            librosa.feature.spectral_rolloff(y=seg, sr=SR, n_fft=FRAME_LEN,
                                              hop_length=HOP_LEN, roll_percent=0.85)
        )
        all_zcr.append(
            librosa.feature.zero_crossing_rate(seg, frame_length=FRAME_LEN, hop_length=HOP_LEN)
        )

    def _cat1d(lst):
        return np.concatenate(lst, axis=1).flatten() if lst else np.array([])

    return {
        'mfcc':      np.concatenate(all_mfcc, axis=1) if all_mfcc else None,
        'energy':    _cat1d(all_energy),
        'centroid':  _cat1d(all_centroid),
        'bandwidth': _cat1d(all_bw),
        'rolloff':   _cat1d(all_rolloff),
        'zcr':       _cat1d(all_zcr),
    }


## ─────────────────────────────────────────────────────────────────────────────
## 4. Features prosódicas y de calidad vocal (Praat via parselmouth)
## ─────────────────────────────────────────────────────────────────────────────

def extract_praat_features(segments: list) -> dict:
    """
    F0 (pitch), HNR, jitter y shimmer via Praat para todos los segmentos.

    F0 (Fundamental Frequency):
        Frecuencia de vibración de las cuerdas vocales. Indicador prosódico
        primario en depresión: reducción de media y variabilidad (Cummins 2015).
        Se usa el estimador SHR de Praat (To Pitch) en lugar de pYIN de librosa
        por ser ~10x más rápido con precisión comparable para habla limpia.
        Solo se contabilizan frames sonoros (F0 > 0).

    HNR (Harmonics-to-Noise Ratio):
        Ratio entre componentes armónicos y ruido en la señal vocal. Mide
        la periodicidad de la voz. Valores más bajos indican mayor afonía/ronquera.
        Los pacientes deprimidos muestran HNR significativamente menor
        (Bhatt et al. 2021).

    Jitter (perturbación de periodo):
        Variación ciclo a ciclo del periodo glotal (inestabilidad de F0).
        Jitter local = |T_i - T_{i-1}| / mean(T). Mayor en voces con
        patología o tensión. Asociado positivamente con depresión.

    Shimmer (perturbación de amplitud):
        Variación ciclo a ciclo de la amplitud del pulso glotal. Indicador
        de irregularidad en la adducción de cuerdas vocales. También se
        eleva en depresión y estados de fatiga vocal.

    Mínimo para calidad vocal (MIN_DUR_VQ = 0.5 s):
        Praat necesita al menos ~3-5 períodos glotales (~30-65 ms a F0 normal)
        para estimar jitter y shimmer de forma estable. 0.5 s es conservador.
    """
    f0_voiced   = []
    n_total     = 0
    n_voiced    = 0
    hnr_list    = []
    jitter_list = []
    shimmer_list = []

    for seg in segments:
        if len(seg) < FRAME_LEN:
            continue

        snd = parselmouth.Sound(seg.astype(np.float64), sampling_frequency=SR)

        # F0 (Praat pitch tracker, SHR method)
        try:
            pitch_obj = snd.to_pitch(
                time_step=HOP_LEN / SR,
                pitch_floor=F0_FMIN,
                pitch_ceiling=F0_FMAX
            )
            f0_arr = pitch_obj.selected_array['frequency']
            n_total  += len(f0_arr)
            n_voiced += int((f0_arr > 0).sum())
            f0_voiced.extend(f0_arr[f0_arr > 0].tolist())
        except Exception:
            pass

        # HNR, Jitter, Shimmer (requieren segmento mínimo)
        if len(seg) / SR < MIN_DUR_VQ:
            continue

        try:
            harm = call(snd, "To Harmonicity (cc)", 0.01, F0_FMIN, 0.1, 1.0)
            hnr  = call(harm, "Get mean", 0, 0)
            if np.isfinite(hnr):
                hnr_list.append(hnr)
        except Exception:
            pass

        try:
            pp = call(snd, "To PointProcess (periodic, cc)", F0_FMIN, F0_FMAX)

            j = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            if np.isfinite(j):
                jitter_list.append(j)

            sh = call([snd, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            if np.isfinite(sh):
                shimmer_list.append(sh)
        except Exception:
            pass

    return {
        'f0':          np.array(f0_voiced),
        'voiced_frac': float(n_voiced / n_total) if n_total > 0 else np.nan,
        'hnr':         np.array(hnr_list),
        'jitter':      np.array(jitter_list),
        'shimmer':     np.array(shimmer_list),
    }


## ─────────────────────────────────────────────────────────────────────────────
## 5. Pipeline por sesión
## ─────────────────────────────────────────────────────────────────────────────

def extract_all_acoustic_features(row: pd.Series) -> dict:
    """
    Pipeline completo por sesión: audio + transcript → vector de features.

    Nivel de análisis: sesión (un vector por sesión).
    Decisión: con 186 sesiones, la segmentación fija (p. ej. 5 s) para DL
    inflaría artificialmente el número de muestras pero rompería la
    independencia entre observaciones en CV. El enfoque de funcionales
    sobre toda la habla del paciente es el estándar para regresión
    PHQ-8 con ML clásico (AVEC 2017 baseline, Williamson 2016, Ringeval 2017).

    Produce 45 features acústicas:
        F0 (2 + voiced_frac = 3) + Energía (2) + MFCCs 13×2 (26) +
        Espectrales 4×2 (8) + Calidad vocal 3×2 (6)
    """
    pid = row.get('participant_id', '?')
    try:
        segments = load_patient_segments(row['path_audio'], row['path_transcript'])

        if not segments:
            print(f"  [{pid}] Sin segmentos válidos")
            return _empty_features()

        sp = extract_spectral_features(segments)
        pr = extract_praat_features(segments)

        out = {}

        # F0
        out.update(_functionals(pr['f0'], 'f0'))
        out['f0_voiced_frac'] = pr['voiced_frac']

        # Energía
        out.update(_functionals(sp['energy'], 'energy'))

        # MFCCs (por coeficiente, indexados desde 1)
        if sp['mfcc'] is not None:
            for i in range(sp['mfcc'].shape[0]):
                out.update(_functionals(sp['mfcc'][i], f'mfcc{i+1:02d}'))
        else:
            for i in range(1, N_MFCC + 1):
                out[f"mfcc{i:02d}_mean"] = np.nan
                out[f"mfcc{i:02d}_std"]  = np.nan

        # Espectrales
        out.update(_functionals(sp['centroid'],  'centroid'))
        out.update(_functionals(sp['bandwidth'], 'bandwidth'))
        out.update(_functionals(sp['rolloff'],   'rolloff'))
        out.update(_functionals(sp['zcr'],       'zcr'))

        # Calidad vocal
        out.update(_functionals(pr['hnr'],     'hnr'))
        out.update(_functionals(pr['jitter'],  'jitter'))
        out.update(_functionals(pr['shimmer'], 'shimmer'))

        return out

    except Exception as e:
        print(f"  [{pid}] ERROR: {e}")
        return _empty_features()

## 2. Ventaneo y features por ventana
`to_windows` corta cada segmento de habla del paciente en ventanas de `WIN_DUR` s **dentro
del propio segmento** (no cruza fronteras de enunciado, para no introducir empalmes que
distorsionen F0/jitter). `extract_window_features` calcula sobre UNA ventana las mismas 45
features acústicas que a nivel de sesión.

In [6]:
def to_windows(segments):
    """Corta cada segmento en ventanas de WIN_DUR s sin solape; conserva el resto
    final si dura >= MIN_WIN s. Ventanea DENTRO de cada segmento (no cruza enunciados)."""
    win, hop, minw = int(WIN_DUR * SR), int(WIN_HOP * SR), int(MIN_WIN * SR)
    out = []
    for seg in segments:
        n = len(seg)
        if n < minw:
            continue
        start = 0
        while start < n:
            w = seg[start:start + win]
            if len(w) >= minw:
                out.append(w)
            start += hop
    return out


def extract_window_features(w):
    """Mismas 45 features acústicas que a nivel de sesión, pero sobre UNA ventana."""
    sp = extract_spectral_features([w])
    pr = extract_praat_features([w])
    out = {}
    out.update(_functionals(pr['f0'], 'f0'))
    out['f0_voiced_frac'] = pr['voiced_frac']
    out.update(_functionals(sp['energy'], 'energy'))
    if sp['mfcc'] is not None:
        for i in range(sp['mfcc'].shape[0]):
            out.update(_functionals(sp['mfcc'][i], f'mfcc{i+1:02d}'))
    else:
        for i in range(1, N_MFCC + 1):
            out[f'mfcc{i:02d}_mean'] = np.nan
            out[f'mfcc{i:02d}_std']  = np.nan
    out.update(_functionals(sp['centroid'],  'centroid'))
    out.update(_functionals(sp['bandwidth'], 'bandwidth'))
    out.update(_functionals(sp['rolloff'],   'rolloff'))
    out.update(_functionals(sp['zcr'],       'zcr'))
    out.update(_functionals(pr['hnr'],     'hnr'))
    out.update(_functionals(pr['jitter'],  'jitter'))
    out.update(_functionals(pr['shimmer'], 'shimmer'))
    return out


def session_to_window_rows(row):
    """Una sesión -> lista de filas (una por ventana):
    metadatos + pausas difundidas (contexto de sesión) + acústicas de la ventana."""
    try:
        segments = load_patient_segments(row['path_audio'], row['path_transcript'])
    except Exception as e:
        print(f"  [{row.get('participant_id','?')}] ERROR carga: {e}")
        return []
    rows = []
    for wi, w in enumerate(to_windows(segments)):
        base = {c: row[c] for c in META_COLS}
        base['window_idx'] = wi
        for c in PAUSE_COLS:            # pausas/turnos difundidas
            base[c] = row[c]
        base.update(extract_window_features(w))   # acústicas de la ventana
        rows.append(base)
    return rows

## 3. Extracción sobre toda la muestra

In [7]:
# Puede tardar bastante más que a nivel de sesión (muchas más unidades).
all_rows = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Sesiones"):
    all_rows.extend(session_to_window_rows(row))

df_seg = pd.DataFrame(all_rows)
sizes = df_seg.groupby('participant_id').size()
print(f"Ventanas totales : {len(df_seg)}")
print(f"Sesiones cubiertas: {df_seg['participant_id'].nunique()} / {len(df_sample)}")
print(f"Ventanas por sesión: media {sizes.mean():.1f} | min {sizes.min()} | max {sizes.max()}")

Sesiones: 100%|██████████| 186/186 [09:52<00:00,  3.19s/it]


Ventanas totales : 19132
Sesiones cubiertas: 186 / 186
Ventanas por sesión: media 102.9 | min 8 | max 309


## 4. Control de calidad y NaN
Se descartan ventanas sin contenido sonoro fiable (baja fracción de frames sonoros). El resto
de NaN residuales (p. ej. jitter/shimmer que Praat no pudo estimar en alguna ventana) se
**imputarán dentro del pipeline de modelado** (mediana, por fold, sin fuga) — NO aquí.

In [8]:
n0 = len(df_seg)
df_seg = df_seg[df_seg['f0_voiced_frac'].fillna(0) > VOICED_MIN].reset_index(drop=True)
print(f"Descartadas {n0 - len(df_seg)} ventanas con voiced_frac <= {VOICED_MIN} (silencio/ruido)")
print(f"Ventanas conservadas: {len(df_seg)}")

nan_cols = df_seg.isna().sum()
nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
if len(nan_cols):
    print("\nNaN residuales por columna (a imputar en el modelado):")
    print(nan_cols.head(20).to_string())
else:
    print("\nSin NaN residuales.")
print(f"\nVentanas con algún NaN: {df_seg.isna().any(axis=1).sum()} / {len(df_seg)}")

Descartadas 123 ventanas con voiced_frac <= 0.05 (silencio/ruido)
Ventanas conservadas: 19009

Sin NaN residuales.

Ventanas con algún NaN: 0 / 19009


## 5. Guardado

In [9]:
out_path = path_output + '/df_segment_features_vf.csv'
df_seg.to_csv(out_path, index=False)
print(f"Guardado: {out_path}  —  shape {df_seg.shape}")
print(f"Columnas: {len(df_seg.columns)}  (5 meta + window_idx + 14 pausas + 45 acústicas)")

Guardado: C:/Users/tblxa/Desktop/Master/UNIR_IA_TFE/output/df_segment_features_vf.csv  —  shape (19009, 65)
Columnas: 65  (5 meta + window_idx + 14 pausas + 45 acústicas)
